In [2]:
import torch
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torch.utils.data import DataLoader
# import segmentation_models_pytorch as smp
from gUnet import Unet2D

# ----------------------------
# Load test data
# ----------------------------
def load_test_data(
    sims_path:   str = "data/sample_20_21_fixwin_sims.npy",
    mask_path:   str = "data/sample_20_21_fixwin_mask.npy",
    day_path:    str = "data/sample_20_21_fixwin_day.npy",
    hour_path:   str = "data/sample_20_21_fixwin_hour.npy",
    latlon_path: str = "data/sample_20_21_fixwin_latlon.npy",
    stats_path:  str = "norm_stats.pt",
    test_split:  float = 1.0,
    seed:        int = 42,
):
    # --- other datasets (uncomment to switch) ---
    # sims_path, mask_path, day_path, hour_path, latlon_path = (
    #     "data/sample_16_19_fixwin_sims.npy",
    #     "data/sample_16_19_fixwin_mask.npy",
    #     "data/sample_16_19_fixwin_day.npy",
    #     "data/sample_16_19_fixwin_hour.npy",
    #     "data/sample_16_19_fixwin_latlon.npy",
    # )  # train-period fix window (in-sample eval)
    # sims_path, mask_path, day_path, hour_path, latlon_path = (
    #     "data/sample_2020L_sims.npy",
    #     "data/sample_2020L_mask.npy",
    #     "data/sample_2020L_day.npy",
    #     "data/sample_2020L_hour.npy",
    #     "data/sample_2020L_latlon.npy",
    # )  # random-window 2020 (legacy)
    sims   = torch.from_numpy(np.load(sims_path))    # [N, T, 3, H, W]
    mask   = torch.from_numpy(np.load(mask_path))    # [N, H, W]
    day    = torch.from_numpy(np.load(day_path))     # [N]
    hour   = torch.from_numpy(np.load(hour_path))    # [N]
    latlon = torch.from_numpy(np.load(latlon_path))  # [N, 2]

    valid = ~(mask > 0).any(dim=(1, 2))

    sims   = sims[valid]
    mask   = mask[valid]
    day    = day[valid]
    hour   = hour[valid]
    latlon = latlon[valid]

    N = len(sims)
    rng = np.random.default_rng(seed)
    indices = rng.permutation(N)
    n_test  = int(N * test_split)

    test_idx = indices[:n_test]

    # Load normalization stats from training
    stats = torch.load(stats_path, map_location="cpu")
    mean  = stats["mean"]   # [1, 9, 1, 1]
    std   = stats["std"]    # [1, 9, 1, 1]
    mean_x = mean.repeat(1, 3, 1, 1)  # [1, 9, 1, 1]
    std_x  = std.repeat(1, 3, 1, 1)   # [1, 9, 1, 1]

    day = day / 365.0
    hour = hour / 24.0

    return {
        "sims":   sims[test_idx],    # [N_test, T, 3, H, W]
        "mask":   mask[test_idx],    # [N_test, H, W]
        "day":    day[test_idx],     # [N_test]
        "hour":   hour[test_idx],    # [N_test]
        "latlon": latlon[test_idx],  # [N_test, 2]
        "mean":   mean_x,
        "std":    std_x
    }


# ----------------------------
# Inference
# ----------------------------
def run_inference(
    model:      torch.nn.Module,
    test_data:  dict,
    device:     str = "cuda" if torch.cuda.is_available() else "cpu",
    batch_size: int = 16,
    iD: int = 3
):
    model = model.to(device).eval()

    sims = test_data["sims"][:,:,:iD,:,:]  # [N, T, 3, H, W]
    day =  test_data["day"]
    hour =  test_data["hour"]
    
    # saved mean and std form training
    mean = test_data["mean"]   # [1, 9, 1, 1]
    std  = test_data["std"]    # [1, 9, 1, 1]

    # Build inputs/targets same as training
    x1 = torch.cat([sims[:, 1], sims[:, 2], sims[:, 3]], dim=1).float()  # [N, 9, H, W]
    x1 = (x1 - mean) / (std + 1e-8)

    print(x1.shape)

    # Build and normalize y
    y_mean = mean[:, :iD, :, :]   # [1, 3, 1, 1]
    y_std  = std[:, :iD, :, :]    # [1, 3, 1, 1]
    y = (sims[:, 3] - sims[:, 2]).float()
    y = (y - y_mean) / (y_std + 1e-8)                                # [N, 3, H, W]

    dataset    = torch.utils.data.TensorDataset(x1, y, day, hour)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    preds1,preds2,preds3, targets = [], [], [], []
    criterion = torch.nn.MSELoss()
    total_loss = 0.0

    with torch.no_grad():
        for x_batch, y_batch, d_batch, h_batch in dataloader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            d_batch, h_batch = d_batch.to(device), h_batch.to(device)

            # ── build t exactly as in training ──
            delta = torch.full_like(d_batch, 4.0)
            t = torch.stack([d_batch, h_batch, delta], dim=-1)          # [B, 3]
            t = t.unsqueeze(1).expand(-1, 3, -1).reshape(x_batch.size(0), -1)  # [B, 9]            
            
            #pred1 = model(x_batch,t)
            pred1 = model(x_batch,d_batch+h_batch/365)
            pred1 = pred1 + x_batch[:, -iD:]  # Add last 3 channels (befor) to get actual prediction
            #pred1 = x_batch[:, -iD:] + 1.5 * pred1 - 0.5 * (x_batch[:, -iD:] - x_batch[:, -2*iD:-iD])

            x_batch2 = torch.cat([x_batch[:, iD:iD*2], x_batch[:, iD*2:iD*3], pred1], dim=1)
            #pred2 = model(x_batch2,t)
            pred2 = model(x_batch2,d_batch+h_batch/365+1.0/365)
            pred2 = pred2 + x_batch2[:, -iD:]
            #pred2 = x_batch2[:, -iD:] + 1.5 * pred2 - 0.5 * (x_batch2[:, -iD:]-x_batch2[:, -2*iD:-iD])


            x_batch3 = torch.cat([x_batch[:, 2*iD:3*iD], x_batch2[:, -iD:], pred2], dim=1)
            #pred3 = model(x_batch3,t)   
            pred3 = model(x_batch3,d_batch+h_batch/365+2.0/365)   
            pred3 = pred3 + pred2
            #pred3 = x_batch3[:, -iD:] + 1.5 * pred3 - 0.5 * (x_batch3[:, -iD:]-x_batch3[:, -2*iD:-iD])

            total_loss += criterion(pred3, y_batch).item() * x_batch.size(0)
            preds1.append(pred1.cpu())
            preds2.append(pred2.cpu())
            preds3.append(pred3.cpu())
            targets.append(y_batch.cpu())

    preds1   = torch.cat(preds1,   dim=0)   # [N, 3, H, W]
    preds2   = torch.cat(preds2,   dim=0)   # [N, 3, H, W]
    preds3   = torch.cat(preds3,   dim=0)   # [N, 3, H, W]
    targets = torch.cat(targets, dim=0)   # [N, 3, H, W]
    mse     = total_loss / len(dataset)

    print(f"Test MSE (normalized): {mse:.6f} | RMSE: {mse**0.5:.6f}")

    # Denormalize back to original scale for plotting
    preds_denorm1   = preds1   * (y_std + 1e-8) + y_mean   # [N, 3, H, W]
    preds_denorm2   = preds2   * (y_std + 1e-8) + y_mean   # [N, 3, H, W]
    preds_denorm3   = preds3   * (y_std + 1e-8) + y_mean   # [N, 3, H, W]

    # targets_denorm = targets * (y_std + 1e-8) + y_mean   # [N, 3, H, W]

    # mse_denorm = torch.mean((preds_denorm - targets_denorm) ** 2).item()
    # print(f"Test MSE (original scale): {mse_denorm:.6f} | RMSE: {mse_denorm**0.5:.6f}")

    return sims[:, 1],sims[:, 2],sims[:, 3],sims[:, 4],sims[:, 5],sims[:, 6],preds_denorm1,preds_denorm2,preds_denorm3


def apply_mask(field: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Set masked regions (mask==1) to NaN so imshow skips them."""
    out = field.copy()
    out[mask == 1] = np.nan
    return out

# ----------------------------
# Plot
# ----------------------------
CHANNEL_NAMES = ["U", "V", "SSH"]  # adjust to your fields

def isotropic_spectrum(field):
    H, W = field.shape
    fft = np.fft.rfft2(field)
    power = np.abs(fft) ** 2
    ky = np.fft.fftfreq(H)
    kx = np.fft.rfftfreq(W)
    ky, kx = np.meshgrid(ky, kx, indexing="ij")
    k_mag = np.sqrt(kx ** 2 + ky ** 2)
    k_flat = k_mag.flatten()
    p_flat = power.flatten()
    k_max = min(H, W) // 2
    spectrum = np.zeros(k_max)
    for i in range(k_max - 1):
        mask_k = (k_flat >= i / k_max) & (k_flat < (i + 1) / k_max)
        if np.any(mask_k):
            spectrum[i] = p_flat[mask_k].mean()
    return np.arange(k_max), spectrum


def plot_sample(
    s1: torch.Tensor,
    s2: torch.Tensor,
    s3: torch.Tensor,
    s4: torch.Tensor,
    s5: torch.Tensor,
    s6: torch.Tensor,
    p1: torch.Tensor,
    p2: torch.Tensor,
    p3: torch.Tensor,
    test_data: dict,
    sample_idx: int = 0,
    save_path: str = None,
):
    s1 = s1[sample_idx].numpy()
    s2 = s2[sample_idx].numpy()
    s3 = s3[sample_idx].numpy()
    s4 = s4[sample_idx].numpy()
    s5 = s5[sample_idx].numpy()
    s6 = s6[sample_idx].numpy()
    p1 = p1[sample_idx].numpy()
    p2 = p2[sample_idx].numpy()
    p3 = p3[sample_idx].numpy()
    mask = test_data["mask"][sample_idx].numpy()
    day = test_data["day"][sample_idx].item()
    hour = test_data["hour"][sample_idx].item()
    lat, lon = test_data["latlon"][sample_idx].tolist()

    n_channels = p1.shape[0]

    # Speed magnitude RMSE metrics (U,V)
    S3 = (s3[0] ** 2 + s3[1] ** 2) ** 0.5
    S4 = (s4[0] ** 2 + s4[1] ** 2) ** 0.5
    S5 = (s5[0] ** 2 + s5[1] ** 2) ** 0.5
    S6 = (s6[0] ** 2 + s6[1] ** 2) ** 0.5
    P1 = (p1[0] ** 2 + p1[1] ** 2) ** 0.5
    P2 = (p2[0] ** 2 + p2[1] ** 2) ** 0.5
    P3 = (p3[0] ** 2 + p3[1] ** 2) ** 0.5

    e1e0 = np.sqrt(np.nanmean((S4 - S3) ** 2))
    e1e1 = np.sqrt(np.nanmean((P1 - S4) ** 2))
    e2e0 = np.sqrt(np.nanmean((S5 - S3) ** 2))
    e2e1 = np.sqrt(np.nanmean((P2 - S5) ** 2))
    e3e0 = np.sqrt(np.nanmean((S6 - S3) ** 2))
    e3e1 = np.sqrt(np.nanmean((P3 - S6) ** 2))

    k4, E4 = isotropic_spectrum(S4)
    k5, E5 = isotropic_spectrum(S5)
    k6, E6 = isotropic_spectrum(S6)
    kP1, EP1 = isotropic_spectrum(P1)
    kP2, EP2 = isotropic_spectrum(P2)
    kP3, EP3 = isotropic_spectrum(P3)

    if save_path is not None:
        fig = plt.figure(figsize=(6 * 3, 2 * n_channels * 3))
        fig.suptitle(
            f"Sample {sample_idx} | Day {int(day * 365):3d}, Hour {int(hour * 24):2d}h | "
            f"Lat {lat:.2f}, Lon {lon:.2f} - Time Embeded, AutoReg Error, Euler",
            fontsize=13,
            fontweight="bold",
        )
        gs = gridspec.GridSpec(n_channels * 2, 6, figure=fig, wspace=0.35, hspace=0.4)

        for ch in range(n_channels):
            ss1 = apply_mask(s1[ch], mask)
            ss2 = apply_mask(s2[ch], mask)
            ss3 = apply_mask(s3[ch], mask)
            ss4 = apply_mask(s4[ch], mask)
            ss5 = apply_mask(s5[ch], mask)
            ss6 = apply_mask(s6[ch], mask)
            pp1 = apply_mask(p1[ch], mask)
            pp2 = apply_mask(p2[ch], mask)
            pp3 = apply_mask(p3[ch], mask)

            vmin = np.nanmin([ss1, ss2, ss3, ss4, ss5, ss6, pp1, pp2, pp3])
            vmax = np.nanmax([ss1, ss2, ss3, ss4, ss5, ss6, pp1, pp2, pp3])

            # Top row: ground-truth sequence
            for col, field, title in [
                (0, ss1, "t[n-2]"), (1, ss2, "t[n-1]"), (2, ss3, "t[n]"),
                (3, ss4, "t[n+1]"), (4, ss5, "t[n+2]"), (5, ss6, "t[n+3]"),
            ]:
                ax = fig.add_subplot(gs[2 * ch, col])
                im = ax.imshow(field, vmin=vmin, vmax=vmax, cmap="viridis")
                ax.set_title(f"{CHANNEL_NAMES[ch]} — {title}")
                ax.axis("off")
                plt.colorbar(im, ax=ax, fraction=0.046)

            # Bottom row: spectra (U row) or error labels, then predictions
            if ch == 0:
                axE1 = fig.add_subplot(gs[2 * ch + 1, 0])
                axE1.loglog(k4[1:], E4[1:], label="GT")
                axE1.loglog(kP1[1:], EP1[1:], label="Prd")
                axE1.legend()

                axE2 = fig.add_subplot(gs[2 * ch + 1, 1])
                axE2.loglog(k5[1:], E5[1:], label="S2")
                axE2.loglog(kP2[1:], EP2[1:], label="P2")

                axE3 = fig.add_subplot(gs[2 * ch + 1, 2])
                axE3.loglog(k6[1:], E6[1:], label="S3")
                axE3.loglog(kP3[1:], EP3[1:], label="P3")
            else:
                axE1 = fig.add_subplot(gs[2 * ch + 1, 0])
                axE1.set_title(f"P {e1e0:.3e} - Pred {e1e1:.3e}")
                axE2 = fig.add_subplot(gs[2 * ch + 1, 1])
                axE2.set_title(f"P {e2e0:.3e} - Pred {e2e1:.3e}")
                axE3 = fig.add_subplot(gs[2 * ch + 1, 2])
                axE3.set_title(f"P {e3e0:.3e} - Pred {e3e1:.3e}")

            e0 = np.sqrt(np.nanmean((ss4 - ss3) ** 2))
            e1 = np.sqrt(np.nanmean((pp1 - ss4) ** 2))
            ax3 = fig.add_subplot(gs[2 * ch + 1, 3])
            im3 = ax3.imshow(pp1, vmin=vmin, vmax=vmax, cmap="viridis")
            ax3.set_title(f"{e0:.3e}|{e1:.3e}|{e1 / (e0 + 1e-12):.1e}", fontsize=8)
            ax3.axis("off")
            plt.colorbar(im3, ax=ax3, fraction=0.046)

            e0 = np.sqrt(np.nanmean((ss5 - ss3) ** 2))
            e2 = np.sqrt(np.nanmean((pp2 - ss5) ** 2))
            ax4 = fig.add_subplot(gs[2 * ch + 1, 4])
            im4 = ax4.imshow(pp2, vmin=vmin, vmax=vmax, cmap="viridis")
            ax4.set_title(f"{e0:.3e}|{e2:.3e}|{e2 / (e0 + 1e-12):.1e}", fontsize=8)
            ax4.axis("off")
            plt.colorbar(im4, ax=ax4, fraction=0.046)

            e0 = np.sqrt(np.nanmean((ss6 - ss3) ** 2))
            e3 = np.sqrt(np.nanmean((pp3 - ss6) ** 2))
            ax5 = fig.add_subplot(gs[2 * ch + 1, 5])
            im5 = ax5.imshow(pp3, vmin=vmin, vmax=vmax, cmap="viridis")
            ax5.set_title(f"{e0:.3e}|{e3:.3e}|{e3 / (e0 + 1e-12):.1e}", fontsize=8)
            ax5.axis("off")
            plt.colorbar(im5, ax=ax5, fraction=0.046)

        os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved to {save_path}")

    return e1e0, e1e1, e2e0, e2e1, e3e0, e3e1


# ----------------------------
# Usage
# ----------------------------
# model = DSCMS(in_channels=9, out_channels=3)
# model = smp.Unet(
#     encoder_name="resnet18",
#     in_channels=9,
#     classes=3,
# )
iD = 3

# model = singleWaveUNet(
#     in_channels=3*iD,
#     out_channels=iD,
#     depth=3,                 # variable depth
#     base_channels=32,
#     bottleneck_reduction=1,  # reduced bottleneck
# )

Par = {
    "lb": 3,      # number of past timesteps
    "nf": 3,      # channels per timestep
    "nx": 128,
    "ny": 128,
    "inp_shift": 0.0,
    "inp_scale": 1.0,
    "out_shift": 0.0,
    "out_scale": 1.0,
    "t_shift": 0.0,
    "t_scale": 1.0,
}
model = Unet2D(
    dim=32,
    Par=Par,
    dim_mults=(1, 2, 4, 8),
    channels=Par["lb"] * Par["nf"]  # = 9
)

model.load_state_dict(torch.load("best_model_FixWindow1.pth", map_location="cpu"))

test_data = load_test_data()
#bbefor,befor,preds, targets, mse = run_inference(model, test_data)
s1,s2,s3,s4,s5,s6,p1,p2,p3 = run_inference(model, test_data,iD=iD)

print("=>",s1.shape)

aveP = np.array([0, 0, 0])
aveU = np.array([0, 0, 0])

PNG_DIR = "PNG"
os.makedirs(PNG_DIR, exist_ok=True)

# Plot a few samples — edit this list as needed
PLOT_INDICES = [0, 1, 2, 45, 32]
# PLOT_INDICES = [1, 2, 187, 38, 98, 97, 154]
# PLOT_INDICES = list(range(s1.shape[0]))  # all samples (slow)

for i in PLOT_INDICES:
    p4, u4, p8, u8, p12, u12 = plot_sample(
        s1, s2, s3, s4, s5, s6, p1, p2, p3,
        test_data,
        sample_idx=i,
        save_path=os.path.join(PNG_DIR, f"output_sample_{i}.png"),
    )
    aveP = aveP + np.array([p4, p8, p12])
    aveU = aveU + np.array([u4, u8, u12])

print("Mean persistence RMSE (+1d, +2d, +3d):", np.array(aveP) / len(PLOT_INDICES))
print("Mean model RMSE (+1d, +2d, +3d):", np.array(aveU) / len(PLOT_INDICES))

/tmp/ipykernel_2074658/2587583035.py:386: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_model_FixWindow1.pth", map_location="cpu"))
/t

torch.Size([1000, 9, 128, 128])
Test MSE (normalized): 1.626111 | RMSE: 1.275191
=> torch.Size([1000, 3, 128, 128])
Saved to PNG/output_sample_0.png
Saved to PNG/output_sample_1.png
Saved to PNG/output_sample_2.png
Saved to PNG/output_sample_45.png
Saved to PNG/output_sample_32.png
Mean persistence RMSE (+1d, +2d, +3d): [0.14561117 0.14101976 0.1549845 ]
Mean model RMSE (+1d, +2d, +3d): [0.07932697 0.11212083 0.10663119]


In [2]:
import numpy as np
import matplotlib.pyplot as plt


def isotropic_spectrum(field):

    H, W = field.shape

    # Fourier transform
    fft = np.fft.rfft2(field)

    # Power spectrum
    power = np.abs(fft)**2

    # Wavenumbers
    ky = np.fft.fftfreq(H)
    kx = np.fft.rfftfreq(W)

    ky, kx = np.meshgrid(ky, kx, indexing='ij')

    k_mag = np.sqrt(kx**2 + ky**2)

    # Flatten
    k_flat = k_mag.flatten()
    p_flat = power.flatten()

    # Radial bins
    k_max = min(H, W) // 2

    spectrum = np.zeros(k_max)

    for i in range(k_max - 1):

        mask = (k_flat >= i / k_max) & (k_flat < (i + 1) / k_max)

        if np.any(mask):
            spectrum[i] = p_flat[mask].mean()

    k_bins = np.arange(k_max)

    return k_bins, spectrum


def plot_spectrum(field, label=None):

    k, E = isotropic_spectrum(field)

    plt.loglog(k[1:], E[1:], label=label)

    plt.xlabel("Wavenumber k")
    plt.ylabel("Power Spectrum")

    if label is not None:
        plt.legend()

    plt.grid(True)

In [3]:
import numpy as np

# Inspect 2020–2021 fixed-window test export
sims = np.load("data/sample_20_21_fixwin_sims.npy", mmap_mode="r")
mask = np.load("data/sample_20_21_fixwin_mask.npy", mmap_mode="r")
window = np.load("data/sample_20_21_fixwin_window.npy")
print("sims shape:", sims.shape)
print("mask shape:", mask.shape)
print("window [top, left, h, w, angle]:", window)

sims shape: (1000, 60, 3, 128, 128)
mask shape: (1000, 128, 128)
window [top, left, h, w, angle]: [330.  70. 128. 128. -45.]


In [4]:
# Count valid samples (no invalid pixels in fixed-window mask)
valid = ~(mask > 0).any(axis=(1, 2))
print(f"Valid samples: {valid.sum()} / {len(mask)}")
print(f"Invalid samples: {(~valid).sum()}")

# Legacy random-window check (uncomment if needed):
# m = np.load("data/sample_2020L_sims.npy")
# mask_r = np.load("data/sample_2020L_mask.npy")
# valid_r = ~(mask_r > 0).any(axis=(1, 2))
# print(f"Random-window valid: {valid_r.sum()} / {len(mask_r)}")

Valid samples: 1000 / 1000
Invalid samples: 0
